In [1]:
import ee 
from RadGEEToolbox import GenericCollection, LandsatCollection, get_palette
# import GEE_UBM
from GEE_UBM import InputCollections, build_model_ready_collection, OriginalUBMRun, ModifiedUBM1Run, check_merged_collection
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import os

In [2]:
service_account = 'localpythonscripts@ut-gee-ugs-bsf-dev.iam.gserviceaccount.com'
credentials = ee.ServiceAccountCredentials(service_account, 'C:\\Users\\mradwin\\ut-gee-ugs-bsf-dev-53dcc5d729e0.json')
ee.Initialize(credentials=credentials)

In [3]:
UBM_assets_dict = ee.data.listAssets('projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/')

UBM_assets = []
for dict in UBM_assets_dict['assets']:
    UBM_assets.append(dict['name'])
print(UBM_assets)

UT_boundary = ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry()
GSL_basin = ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Watersheds/Merged_GSL_Basin_Watershed").geometry()
# HUC6 basins
UT_basin_boundaries = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UT_HUC6_Basin_Boundaries')
UT_basin_boundary_names = UT_basin_boundaries.aggregate_array('Name').getInfo()
UT_basin_boundary_names_cleaned = [name.replace(' ', '_') for name in UT_basin_boundary_names]
print(UT_basin_boundary_names)

# HUC8 watersheds
UT_watershed_boundaries = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/Utah_Watersheds/Utah_Regional_Watersheds')
UT_watershed_names = UT_watershed_boundaries.aggregate_array('HU_8_NAME').distinct().getInfo()
UT_watershed_names_cleaned = [name.replace(' ', '_').replace('-', '_').replace("'", '').replace(",", '') for name in UT_watershed_names]
print(UT_watershed_names)

['projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/Mod_UBM_1_RF1kmST_POLPor_OLMFC_HHSWP_NGMDGKSdM_DAYMETSNOM_ETDALEXI_IRRIm_M_mm', 'projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/Mod_UBM_1_RF1kmST_POLPor_OLMFC_HHSWP_NGMDGKSdM_DAYMETSNOM_ETEMTRIC_IRRIm_M_mm', 'projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/Mod_UBM_1_RF1kmST_POLPor_OLMFC_HHSWP_NGMDGKSdM_DAYMETSNOM_ETGSEBAL_IRRIm_M_mm', 'projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/Mod_UBM_1_RF1kmST_POLPor_OLMFC_HHSWP_NGMDGKSdM_DAYMETSNOM_ETPTJPL_IRRIm_M_mm', 'projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/Mod_UBM_1_RF1kmST_POLPor_OLMFC_HHSWP_NGMDGKSdM_DAYMETSNOM_ETSBOP_IRRIm_M_mm', 'projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/Mod_UBM_1_RF1kmST_POLPor_OLMFC_HHSWP_NGMDGKSdM_DAYMETSNOM_ETSIMS_IRRIm_M_mm', 'projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/Mod_UBM_1_RF1kmST_POLPor_OLMFC_HHSWP_NGMDGKSdM_GRIDMETSNOM_ETDALEXI_IRRIm_M_mm', 'projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1Runs/Mod_UBM_1_RF1kmST_P

In [4]:
### Creating folders for saving HUC6 basin zonal stats ###
huc6_output_directory = 'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Zonal_Stats_Timeseries\\HUC6_Basins'
create_huc6_folders = False
if create_huc6_folders:
    for basin_name in UT_basin_boundary_names_cleaned:
        folder_name = basin_name #.replace(' ', '_')
        folder_path = os.path.join(huc6_output_directory, folder_name)
        os.makedirs(folder_path, exist_ok=True)

### Creating folders for saving HUC8 basin zonal stats ###
huc8_output_directory = 'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Zonal_Stats_Timeseries\\HUC8_Watersheds'
create_huc8_folders = False
if create_huc8_folders:
    for basin_name in UT_watershed_names_cleaned:
        folder_name = basin_name #.replace(' ', '_').replace('-', '_').replace("'", '')
        folder_path = os.path.join(huc8_output_directory, folder_name)
        os.makedirs(folder_path, exist_ok=True)

In [5]:
def convert_values_to_volume(image):
    def convert_depth_to_volume(image):
        """Converts pixel values from depth (mm) to volume (m^3)."""
        pixel_area = ee.Image.pixelArea()
        depth_in_meters = image.multiply(0.001)
        volume_m3 = pixel_area.multiply(depth_in_meters)
        return volume_m3.copyProperties(image, image.propertyNames())
    inputs = image.select(['precip_and_snowmelt_input', 'irrigation', 'AET', 'Runoff', 'Recharge', 'Soil_Water_End_Of_Previous_Timestep'])
    other_bands = image.select(['soil_thickness', 'soil_porosity', 'field_capacity', 'wilting_point', 'Geo_K', 'Soil_Saturation_Percent_End_Of_Timestep'])
    inputs_as_volume = ee.Image(convert_depth_to_volume(inputs)).rename(['precip_and_snowmelt_input_m3', 'irrigation_m3', 'AET_m3', 'Runoff_m3', 'Recharge_m3', 'Soil_Water_End_Of_Previous_Timestep_m3'])
    native_proj = image.projection()
    new_img = inputs_as_volume.addBands(ee.Image(other_bands)).setDefaultProjection(native_proj).copyProperties(image, image.propertyNames())
    return new_img

In [6]:
years = list(range(2005, 2025, 1))
print(years)

[2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [6]:
for model_run in UBM_assets:
    UBM_input_volume = ee.ImageCollection(model_run).map(convert_values_to_volume).select(['precip_and_snowmelt_input_m3', 'irrigation_m3', 'AET_m3', 'Runoff_m3', 'Recharge_m3', 'Soil_Water_End_Of_Previous_Timestep_m3'])
    UBM_input_depth = ee.ImageCollection(model_run)
    UBM_collection = UBM_input_depth.combine(UBM_input_volume)
    if 'GSEBAL' in model_run:
        ET_dataset = 'GEESEBAL'
    elif 'DALEXI' in model_run:
        ET_dataset = 'DisALEXI'
    elif 'EMTRIC' in model_run:
        ET_dataset = 'EEMETRIC'
    elif 'PTJPL' in model_run:
        ET_dataset = 'PTJPL'
    elif 'SBOP' in model_run:
        ET_dataset = 'SSEBOP'
    elif 'SIMS' in model_run:
        ET_dataset = 'SIMS'

    if 'DAYMET' in model_run:
        precip_dataset = 'DAYMET'
    elif 'GRIDMET' in model_run:
        precip_dataset = 'GRIDMET'
    elif 'PRISM' in model_run:
        precip_dataset = 'PRISM'
    
    # multiband_collection = GenericCollection(collection=UBM_collection)
    # print(multiband_collection.image_grab(-1).bandNames().getInfo())
    # print(multiband_collection.image_grab(-1).projection().nominalScale().getInfo())
    for basin_name, cleaned_basin_name in zip(UT_basin_boundary_names, UT_basin_boundary_names_cleaned):
        basin_geometry = UT_basin_boundaries.filter(ee.Filter.eq('Name', basin_name)).geometry()
        years = list(range(2005, 2025, 1))
        basin_dfs = []
        for year in years:
            start_date = f'{year}-01-01'
            end_date = f'{year}-12-31'
            multiband_collection = GenericCollection(collection=UBM_collection, start_date=start_date, end_date=end_date)
            multiband_stats = multiband_collection.multiband_zonal_stats(geometry=basin_geometry, bands=['precip_and_snowmelt_input', 'irrigation', 'AET', 
                                                                                                'soil_thickness', 'soil_porosity', 'field_capacity', 'wilting_point', 
                                                                                                'Geo_K', 'Runoff', 'Recharge', 'Soil_Water_End_Of_Previous_Timestep', 'Soil_Saturation_Percent_End_Of_Timestep',
                                                                                                'precip_and_snowmelt_input_m3', 'irrigation_m3', 'AET_m3', 'Runoff_m3', 'Recharge_m3', 'Soil_Water_End_Of_Previous_Timestep_m3'],
                                                                        reducer_types=['mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean',
                                                                                    'sum', 'sum', 'sum', 'sum', 'sum', 'sum'],
                                                                        scale=multiband_collection.image_grab(-1).projection().nominalScale().getInfo(),
                                                                        geometry_name=basin_name,
                                                                        include_area=True,
                                                                        tileScale=4)
            multiband_stats = multiband_stats.reset_index()
            basin_dfs.append(multiband_stats)
            print(f'Finished processing {cleaned_basin_name}_{precip_dataset}_{ET_dataset}_{start_date}_{end_date}')
        master_basin_df = pd.concat(basin_dfs, axis=0)
        master_basin_df['Date'] = pd.to_datetime(master_basin_df['Date'])
        master_basin_df = master_basin_df.sort_values(by='Date').reset_index(drop=True)
        master_basin_df.to_csv(f'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Zonal_Stats_Timeseries\\HUC6_Basins\\{cleaned_basin_name}\\{cleaned_basin_name}_{precip_dataset}_{ET_dataset}_zs.csv')
    for basin_name, cleaned_basin_name in zip(UT_watershed_names, UT_watershed_names_cleaned):
        basin_geometry = UT_watershed_boundaries.filter(ee.Filter.eq('HU_8_NAME', basin_name)).geometry()
        years = list(range(2005, 2025, 1))
        watershed_dfs = []
        for year in years:
            start_date = f'{year}-01-01'
            end_date = f'{year}-12-31'
            multiband_stats = multiband_collection.multiband_zonal_stats(geometry=basin_geometry, bands=['precip_and_snowmelt_input', 'irrigation', 'AET', 
                                                                                                'soil_thickness', 'soil_porosity', 'field_capacity', 'wilting_point', 
                                                                                                'Geo_K', 'Runoff', 'Recharge', 'Soil_Water_End_Of_Previous_Timestep', 'Soil_Saturation_Percent_End_Of_Timestep',
                                                                                                'precip_and_snowmelt_input_m3', 'irrigation_m3', 'AET_m3', 'Runoff_m3', 'Recharge_m3', 'Soil_Water_End_Of_Previous_Timestep_m3'],
                                                                        reducer_types=['mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean',
                                                                                    'sum', 'sum', 'sum', 'sum', 'sum', 'sum'],
                                                                        scale=multiband_collection.image_grab(-1).projection().nominalScale().getInfo(),
                                                                        geometry_name=basin_name,
                                                                        include_area=True,
                                                                        tileScale=4)
            multiband_stats = multiband_stats.reset_index()
            watershed_dfs.append(multiband_stats)
            print(f'Finished processing {cleaned_basin_name}_{precip_dataset}_{ET_dataset}_{start_date}_{end_date}')
        master_watershed_df = pd.concat(watershed_dfs, axis=0)
        master_watershed_df['Date'] = pd.to_datetime(master_watershed_df['Date'])
        master_watershed_df = master_watershed_df.sort_values(by='Date').reset_index(drop=True)
        master_watershed_df.to_csv(f'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Zonal_Stats_Timeseries\\HUC8_Watersheds\\{cleaned_basin_name}\\{cleaned_basin_name}_{precip_dataset}_{ET_dataset}_zs.csv')




Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2005-01-01_2005-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2006-01-01_2006-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2007-01-01_2007-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2008-01-01_2008-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2009-01-01_2009-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2010-01-01_2010-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2011-01-01_2011-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2012-01-01_2012-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2013-01-01_2013-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2014-01-01_2014-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2015-01-01_2015-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2016-01-01_2016-12-31
Finished processing Colorado_Headwaters_DAYMET_DisALEXI_2017-01-

In [9]:
for model_run in UBM_assets:
    UBM_input_volume = ee.ImageCollection(model_run).map(convert_values_to_volume).select(['precip_and_snowmelt_input_m3', 'irrigation_m3', 'AET_m3', 'Runoff_m3', 'Recharge_m3', 'Soil_Water_End_Of_Previous_Timestep_m3'])
    UBM_input_depth = ee.ImageCollection(model_run)
    UBM_collection = UBM_input_depth.combine(UBM_input_volume)
    if 'GSEBAL' in model_run:
        ET_dataset = 'GEESEBAL'
    elif 'DALEXI' in model_run:
        ET_dataset = 'DisALEXI'
    elif 'EMTRIC' in model_run:
        ET_dataset = 'EEMETRIC'
    elif 'PTJPL' in model_run:
        ET_dataset = 'PTJPL'
    elif 'SBOP' in model_run:
        ET_dataset = 'SSEBOP'
    elif 'SIMS' in model_run:
        ET_dataset = 'SIMS'

    if 'DAYMET' in model_run:
        precip_dataset = 'DAYMET'
    elif 'GRIDMET' in model_run:
        precip_dataset = 'GRIDMET'
    elif 'PRISM' in model_run:
        precip_dataset = 'PRISM'
    
    # multiband_collection = GenericCollection(collection=UBM_collection)
    # print(multiband_collection.image_grab(-1).bandNames().getInfo())
    # print(multiband_collection.image_grab(-1).projection().nominalScale().getInfo())
    for basin, cleaned_basin_name in zip([UT_boundary, GSL_basin], ['Utah_Statewide', 'GSL_Basin_Watershed']):
        basin_geometry = basin
        years = list(range(2005, 2025, 1))
        basin_dfs = []
        for year in years:
            start_date = f'{year}-01-01'
            end_date = f'{year}-12-31'
            multiband_collection = GenericCollection(collection=UBM_collection, start_date=start_date, end_date=end_date)
            multiband_stats = multiband_collection.multiband_zonal_stats(geometry=basin_geometry, bands=['precip_and_snowmelt_input', 'irrigation', 'AET', 
                                                                                                'soil_thickness', 'soil_porosity', 'field_capacity', 'wilting_point', 
                                                                                                'Geo_K', 'Runoff', 'Recharge', 'Soil_Water_End_Of_Previous_Timestep', 'Soil_Saturation_Percent_End_Of_Timestep',
                                                                                                'precip_and_snowmelt_input_m3', 'irrigation_m3', 'AET_m3', 'Runoff_m3', 'Recharge_m3', 'Soil_Water_End_Of_Previous_Timestep_m3'],
                                                                        reducer_types=['mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean',
                                                                                    'sum', 'sum', 'sum', 'sum', 'sum', 'sum'],
                                                                        scale=multiband_collection.image_grab(-1).projection().nominalScale().getInfo(),
                                                                        geometry_name=cleaned_basin_name,
                                                                        include_area=True,
                                                                        tileScale=4)
            multiband_stats = multiband_stats.reset_index()
            basin_dfs.append(multiband_stats)
            print(f'Finished processing {cleaned_basin_name}_{precip_dataset}_{ET_dataset}_{start_date}_{end_date}')
        master_basin_df = pd.concat(basin_dfs, axis=0)
        master_basin_df['Date'] = pd.to_datetime(master_basin_df['Date'])
        master_basin_df = master_basin_df.sort_values(by='Date').reset_index(drop=True)
        master_basin_df.to_csv(f'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Zonal_Stats_Timeseries\\HUC6_Basins\\{cleaned_basin_name}\\{cleaned_basin_name}_{precip_dataset}_{ET_dataset}_zs.csv')


Finished processing Utah_Statewide_DAYMET_DisALEXI_2005-01-01_2005-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2006-01-01_2006-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2007-01-01_2007-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2008-01-01_2008-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2009-01-01_2009-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2010-01-01_2010-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2011-01-01_2011-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2012-01-01_2012-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2013-01-01_2013-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2014-01-01_2014-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2015-01-01_2015-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2016-01-01_2016-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_2017-01-01_2017-12-31
Finished processing Utah_Statewide_DAYMET_DisALEXI_

In [7]:
big_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/big_creek').geometry()
blacks_fork = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/blacks_fork').geometry()
castle_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/castle_creek').geometry()
farmington_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/farmington_creek').geometry()
mammoth_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/mammoth_creek').geometry()
muddy_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/muddy_creek').geometry()
weber_river = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/weber_river').geometry()
west_canyon = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/west_canyon').geometry()
whiterocks_uinta = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/whiterocks_uinta').geometry()

basin_geometries_dict = {
    "big_creek": big_creek,
    "blacks_fork": blacks_fork,
    "castle_creek": castle_creek,
    "farmington_creek": farmington_creek,
    "mammoth_creek": mammoth_creek,
    "muddy_creek": muddy_creek,
    "weber_river": weber_river,
    "west_canyon": west_canyon,
    "whiterocks_uinta": whiterocks_uinta
}

basins_list = [big_creek, blacks_fork, castle_creek, farmington_creek, mammoth_creek, muddy_creek, weber_river, west_canyon, whiterocks_uinta]
basins_names = ['Big_Creek', 'Blacks_Fork', 'Castle_Creek', 'Farmington_Creek', 'Mammoth_Creek', 'Muddy_Creek', 'Weber_River', 'West_Canyon', 'Whiterocks_Uinta']

In [11]:
for model_run in UBM_assets:
    UBM_input_volume = ee.ImageCollection(model_run).map(convert_values_to_volume).select(['precip_and_snowmelt_input_m3', 'irrigation_m3', 'AET_m3', 'Runoff_m3', 'Recharge_m3', 'Soil_Water_End_Of_Previous_Timestep_m3'])
    UBM_input_depth = ee.ImageCollection(model_run)
    UBM_collection = UBM_input_depth.combine(UBM_input_volume)
    if 'GSEBAL' in model_run:
        ET_dataset = 'GEESEBAL'
    elif 'DALEXI' in model_run:
        ET_dataset = 'DisALEXI'
    elif 'EMTRIC' in model_run:
        ET_dataset = 'EEMETRIC'
    elif 'PTJPL' in model_run:
        ET_dataset = 'PTJPL'
    elif 'SBOP' in model_run:
        ET_dataset = 'SSEBOP'
    elif 'SIMS' in model_run:
        ET_dataset = 'SIMS'

    if 'DAYMET' in model_run:
        precip_dataset = 'DAYMET'
    elif 'GRIDMET' in model_run:
        precip_dataset = 'GRIDMET'
    elif 'PRISM' in model_run:
        precip_dataset = 'PRISM'
    
    # multiband_collection = GenericCollection(collection=UBM_collection)
    # print(multiband_collection.image_grab(-1).bandNames().getInfo())
    # print(multiband_collection.image_grab(-1).projection().nominalScale().getInfo())
    # for basin_name, cleaned_basin_name in zip(UT_basin_boundary_names, UT_basin_boundary_names_cleaned):
    for basin, basin_name in zip(basins_list, basins_names):
        if ET_dataset == 'SIMS':
            continue
        else:
            if os.path.exists(f'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Zonal_Stats_Timeseries\\Validation_Zonal_Stats\\Calibrated_Basin_Timeseries\\Ensemble_Runs\\{basin_name}\\{basin_name}_{precip_dataset}_{ET_dataset}_zs.csv'):
                continue
            else:
                basin_geometry = basin
                years = list(range(2005, 2025, 1))
                basin_dfs = []
                for year in years:
                    start_date = f'{year}-01-01'
                    end_date = f'{year}-12-31'
                    multiband_collection = GenericCollection(collection=UBM_collection, start_date=start_date, end_date=end_date)
                    multiband_stats = multiband_collection.multiband_zonal_stats(geometry=basin_geometry, bands=['precip_and_snowmelt_input', 'irrigation', 'AET', 
                                                                                                        'soil_thickness', 'soil_porosity', 'field_capacity', 'wilting_point', 
                                                                                                        'Geo_K', 'Runoff', 'Recharge', 'Soil_Water_End_Of_Previous_Timestep', 'Soil_Saturation_Percent_End_Of_Timestep',
                                                                                                        'precip_and_snowmelt_input_m3', 'irrigation_m3', 'AET_m3', 'Runoff_m3', 'Recharge_m3', 'Soil_Water_End_Of_Previous_Timestep_m3'],
                                                                                reducer_types=['mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean', 'mean',
                                                                                            'sum', 'sum', 'sum', 'sum', 'sum', 'sum'],
                                                                                scale=multiband_collection.image_grab(-1).projection().nominalScale().getInfo(),
                                                                                geometry_name=basin_name,
                                                                                include_area=True,
                                                                                tileScale=4)
                    multiband_stats = multiband_stats.reset_index()
                    basin_dfs.append(multiband_stats)
                    print(f'Finished processing {basin_name}_{precip_dataset}_{ET_dataset}_{start_date}_{end_date}')
                master_basin_df = pd.concat(basin_dfs, axis=0)
                master_basin_df['Date'] = pd.to_datetime(master_basin_df['Date'])
                master_basin_df = master_basin_df.sort_values(by='Date').reset_index(drop=True)
                os.makedirs(f'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Zonal_Stats_Timeseries\\Validation_Zonal_Stats\\Calibrated_Basin_Timeseries\\Ensemble_Runs\\{basin_name}', exist_ok=True)
                master_basin_df.to_csv(f'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Zonal_Stats_Timeseries\\Validation_Zonal_Stats\\Calibrated_Basin_Timeseries\\Ensemble_Runs\\{basin_name}\\{basin_name}_{precip_dataset}_{ET_dataset}_zs.csv')

Finished processing Blacks_Fork_PRISM_EEMETRIC_2005-01-01_2005-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2006-01-01_2006-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2007-01-01_2007-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2008-01-01_2008-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2009-01-01_2009-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2010-01-01_2010-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2011-01-01_2011-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2012-01-01_2012-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2013-01-01_2013-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2014-01-01_2014-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2015-01-01_2015-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2016-01-01_2016-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2017-01-01_2017-12-31
Finished processing Blacks_Fork_PRISM_EEMETRIC_2018-01-01_2018-12-31
Finished processing Blacks_Fork_PR